# Lab 0 — Exploring unlabelled data

Ungraded introductory practice. Download the full customer CSV and put it beside this notebook. It contains six synthetic customers with age, income, spending score and purchase frequency, without supplied segment labels. The two-feature environmental, customer, student study-habit and music-listening CSVs from the website also work. Install pandas and matplotlib if needed.

Goal: inspect features, investigate missing values and an unusual observation, compare scales, then justify an approach. Your output is this notebook with plots and short written interpretations. Do not fit K-means yet.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

dataset_file = "customers-full-lab0.csv"  # Or: environment-lab0.csv / customers-lab0.csv / students-lab0.csv / music-lab0.csv
df = pd.read_csv(dataset_file)
feature_cols = list(df.columns[1:])
X_original = df[feature_cols].apply(pd.to_numeric, errors="coerce")
display(df)
display(X_original.describe())
display(X_original.isna().sum().rename("missing_count"))


## 1. Identify the observations and features
What does one row represent? Why is sample_id not a measurement or known output label? Which feature has a missing value?

In [ ]:
# Select any two features for a plot; this does not discard other features from the table.
plot_x = feature_cols[1] if len(feature_cols) > 2 else feature_cols[0]
plot_y = feature_cols[2] if len(feature_cols) > 2 else feature_cols[1]
ax = X_original.plot.scatter(x=plot_x, y=plot_y, figsize=(7, 4))
ax.set_title("Original synthetic observations")
plt.show()


## 2. Make one preparation decision at a time
The last row is preselected for inspection in these examples. Removing the final row is not a general outlier rule. Keep it unless you can justify exclusion. Try the choices below. The two-feature CSVs reproduce the browser experiment; the full customer CSV extends it to four features.


In [ ]:
include_unusual = True
missing_strategy = "leave"  # "leave", "median", or "drop"
standardise = False

X = X_original.copy()
if not include_unusual:
    X = X.iloc[:-1].copy()  # Example-specific inspection choice, not automatic cleaning.

medians = X.median()
if missing_strategy == "median":
    X = X.fillna(medians)
elif missing_strategy == "drop":
    X = X.dropna()

# Fit scaling statistics on retained complete observations, matching the website.
complete = X.dropna()
means = complete.mean()
stds = complete.std(ddof=0)  # Population standard deviation.
prepared = (X - means) / stds.replace(0, 1) if standardise else X.copy()
display(prepared)
print("Retained rows:", len(prepared), "Plottable rows:", len(prepared.dropna()))


## 3. Compare numerical distance contributions
The plot automatically fits each axis, so apparent screen distances are not feature-space distances. Compare the squared feature differences explicitly.

In [ ]:
gaps_squared = (prepared.iloc[0] - prepared.iloc[1]) ** 2
total = gaps_squared.sum()
comparison = pd.DataFrame({"squared_gap": gaps_squared, "contribution_percent": 100 * gaps_squared / total})
display(comparison)
print("Squared Euclidean distance:", total)
ax = prepared.plot.scatter(x=plot_x, y=plot_y, figsize=(7, 4))
ax.set_title("Prepared observations" + (" (z-scores)" if standardise else " (original units)"))
plt.show()


## 4. Explain your findings
1. What changed when you filled or dropped the incomplete row? What information or uncertainty remains?
2. Is the unusual observation necessarily wrong? What context would you need?
3. Which feature dominated the first two observations’ squared distance before and after standardisation?
4. What do you want to discover in this dataset? Recommend a technique family and justify it.
5. State two limitations or checks before interpreting a discovered pattern.

Your written explanation matters; lower error or a neat picture does not prove a meaningful grouping.

In [ ]:
# Optional: save the prepared values with their sample IDs.
prepared_with_ids = df.loc[prepared.index, ["sample_id"]].join(prepared)
prepared_with_ids.to_csv("lab0-prepared.csv", index=False)


## Optional: encoding a categorical feature
This separate soil example illustrates one-hot encoding of nominal categories. IDs are excluded; the encoded columns are still input features. It does not change the customer or environmental data above.


In [ ]:
soil = pd.DataFrame({"soil_type": ["Sand", "Clay", "Rock"]})
display(pd.get_dummies(soil, columns=["soil_type"], dtype=int))
